# 01 — Synthetic chaos and the RECD

**Labels:** `[OPERACIONAL]` demo on synthetic coupled logistic maps.  
Not a field claim (`[EMPÍRICO]`). Thresholds are operational defaults.

This notebook:
1. Generates multivariate logistic maps (pre-chaos vs chaos)
2. Computes Systemic Tau τₛ (window = 13)
3. Accumulates discrete extramental time (RECD)
4. Plots τₛ, gate g(τₛ), and T_RECD

**Repo:** https://github.com/johelpadilla/systemic-tau-formal · DOI: https://doi.org/10.5281/zenodo.21516060


## Setup


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if (ROOT / "python" / "core").is_dir():
    pass
elif (ROOT.parent / "python" / "core").is_dir():
    ROOT = ROOT.parent
else:
    # walk up
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "python" / "core").is_dir():
            ROOT = p
            break

sys.path.insert(0, str(ROOT / "python"))
from core import (
    THETA_CHAOS, THETA_STABLE, DELTA,
    compute_taus, accumulate_time, gate_function,
)
print("ROOT =", ROOT)
print("δ =", DELTA, "τ_ch =", THETA_CHAOS, "τ_st =", THETA_STABLE)

## Coupled logistic generator


In [ ]:
def coupled_logistic(T=800, N=4, r=3.85, eps=0.05, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.uniform(0.1, 0.9, size=(T, N))
    for t in range(1, T):
        mean = X[t - 1].mean()
        for i in range(N):
            x = (1 - eps) * X[t - 1, i] + eps * mean
            X[t, i] = r * x * (1 - x)
    return X

def analyze(r, name, seed=42):
    X = coupled_logistic(r=r, seed=seed)
    tg, _ = compute_taus(X, window_size=13)
    T, dtk, g, depth = accumulate_time(tg, window_size=13)
    valid = tg[~np.isnan(tg)]
    return {
        "name": name, "r": r, "X": X, "tau": tg, "T": T, "g": g, "depth": depth,
        "mean_tau": float(valid.mean()),
        "frac_chaos": float(np.mean(np.abs(valid) < THETA_CHAOS)),
        "T_end": float(T[-1]),
        "max_k": int(depth.max()),
    }

## Pre-chaos vs chaos


In [ ]:
results = [analyze(3.30, "pre-chaos"), analyze(3.85, "chaos")]
for res in results:
    print(f"r={res['r']:.2f} ({res['name']})")
    print(f"  mean τₛ        = {res['mean_tau']:.4f}")
    print(f"  frac |τ|<0.41  = {res['frac_chaos']:.3f}")
    print(f"  T_RECD end     = {res['T_end']:.6f}")
    print(f"  max depth k    = {res['max_k']}")
    print()
assert gate_function(0.8) == 1.0
print("Gate sanity OK.")

## Figures


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex="col")
for col, res in enumerate(results):
    t = np.arange(len(res["tau"]))
    axes[0, col].plot(t, res["tau"], lw=0.8, color="C0")
    axes[0, col].axhline(THETA_CHAOS, color="C3", ls="--", lw=0.8, label="±τ_ch")
    axes[0, col].axhline(-THETA_CHAOS, color="C3", ls="--", lw=0.8)
    axes[0, col].axhline(THETA_STABLE, color="C2", ls=":", lw=0.8, label="τ_st")
    axes[0, col].set_title(f"τₛ — r={res['r']} ({res['name']})")
    axes[0, col].set_ylabel("τₛ")
    axes[0, col].legend(fontsize=8, loc="lower right")

    axes[1, col].plot(t, res["g"], lw=0.8, color="C1")
    axes[1, col].set_title("gate g(τₛ)")
    axes[1, col].set_ylabel("g")

    axes[2, col].plot(t, res["T"], lw=1.0, color="C4")
    axes[2, col].set_title("accumulated T_RECD")
    axes[2, col].set_xlabel("time index")
    axes[2, col].set_ylabel("T")

fig.suptitle("[OPERACIONAL] Coupled logistic maps — Systemic Tau + RECD", y=1.01)
fig.tight_layout()
plt.show()

## How to read this

- Low |τₛ| spends more time in the chaotic band of the operational gate.
- RECD depth \(k\) and increments \(\delta^{-k}\) amplify runs inside that band.
- Pre-chaos vs chaos differences here are **illustrative**; quantitative claims need the experimental protocol.

See `docs/FALSIFIABLE_PREDICTIONS.md` and `docs/EXPERIMENTAL_PROTOCOL.md`.
